# Clase 04 — Probabilidad para el analista

**Curso:** Estadística Aplicada con Python · **Bloque 2:** Probabilidad
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/04_probabilidad_analista.md`

## Qué vas a ver en esta clase

- La probabilidad como **frecuencia a largo plazo**, comprobada por simulación.
- **Eventos**, espacio muestral, complementario, unión e intersección.
- **Independencia** y **probabilidad condicional** (y por qué P(A|B) no es P(B|A)).
- La **ley de la probabilidad total** y el **teorema de Bayes** aplicados a un test diagnóstico.
- La **simulación de Monte Carlo** comparada con el cálculo exacto.

## Requisitos

- Clases 1 a 3 (variabilidad de muestreo, frecuencias y distribuciones empíricas).
- Librerías: `numpy`, `pandas`, `matplotlib` (`pip install numpy pandas matplotlib`; ya vienen instaladas en Google Colab).

> **Cómo usar este notebook:** ejecuta las celdas en orden. Antes de cada experimento escribe qué esperas que ocurra; después compara con lo que ves. Todos los números del caso del test diagnóstico son un *ejemplo ilustrativo*, no datos reales.

---
## Celda 0 — Setup

Importamos las librerías y fijamos la **semilla** del generador aleatorio: con la misma semilla, todos obtenemos los mismos números.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerías cargadas correctamente.")

---
## 1. La probabilidad como frecuencia a largo plazo

Lanzamos una moneda equilibrada 100 000 veces y vamos calculando, tras cada lanzamiento, la **frecuencia relativa** de caras. La idea: la probabilidad de un evento es la proporción de veces que ocurre si repites el experimento muchísimas veces.

In [ ]:
N_LANZ = 100_000
lanzamientos = rng.random(N_LANZ) < 0.5                     # True = cara
frec_acum = np.cumsum(lanzamientos) / np.arange(1, N_LANZ + 1)

filas = [{"lanzamientos": n,
          "frecuencia de caras": frec_acum[n - 1],
          "distancia a 0,5": abs(frec_acum[n - 1] - 0.5)}
         for n in [10, 100, 1_000, 10_000, 100_000]]
display(pd.DataFrame(filas).set_index("lanzamientos").round(4))

# cinco trayectorias independientes de 2 000 lanzamientos
for _ in range(5):
    tray = np.cumsum(rng.random(2_000) < 0.5) / np.arange(1, 2_001)
    plt.plot(np.arange(1, 2_001), tray)
plt.axhline(0.5, color="black", linestyle="--")
plt.xscale("log")
plt.title("Frecuencia relativa de caras: cinco experimentos")
plt.xlabel("Número de lanzamientos (escala log)"); plt.ylabel("Frecuencia de caras")
plt.show()

**¿Qué observas?**

- Al principio (pocos lanzamientos) las trayectorias se dispersan mucho. ¿Qué pasa con la dispersión al crecer el número de lanzamientos?
- ¿Hacia qué valor convergen las cinco? ¿Llegan *exactamente* a 0,5?
- ¿Qué te dice esto sobre lo que significa "la probabilidad de cara es 0,5" para un único lanzamiento?

*(Tu respuesta aquí.)*

---
## 2. Espacio muestral y eventos

Con un dado de seis caras, el **espacio muestral** es Ω = {1, 2, 3, 4, 5, 6}. Definimos dos eventos: A = "sale par" y B = "sale más de 4". Calculamos las probabilidades **exactas** contando casos (todos los resultados son igual de probables) y las comprobamos con una simulación.

In [ ]:
omega = np.arange(1, 7)
A = omega[omega % 2 == 0]          # par: {2, 4, 6}
B = omega[omega > 4]               # mayor que 4: {5, 6}

def prob(evento):
    """Probabilidad exacta contando casos (resultados equiprobables)."""
    return len(evento) / len(omega)

union = np.union1d(A, B)
interseccion = np.intersect1d(A, B)

print(f"P(A) = {prob(A):.4f}    P(B) = {prob(B):.4f}")
print(f"P(no A) = 1 − P(A)            = {1 - prob(A):.4f}")
print(f"P(A ∪ B) = {prob(union):.4f}    P(A ∩ B) = {prob(interseccion):.4f}")
print(f"Regla de la suma: P(A) + P(B) − P(A ∩ B) = {prob(A) + prob(B) - prob(interseccion):.4f}")

# Simulación: 200 000 tiradas
tiradas = rng.integers(1, 7, size=200_000)
en_A, en_B = np.isin(tiradas, A), np.isin(tiradas, B)
print()
print(f"Simulación: P(A) ≈ {en_A.mean():.4f}   P(B) ≈ {en_B.mean():.4f}   "
      f"P(A ∪ B) ≈ {(en_A | en_B).mean():.4f}   P(A ∩ B) ≈ {(en_A & en_B).mean():.4f}")

**Observa:** la simulación se acerca a los valores exactos pero no los reproduce al céntimo. Fíjate en que la regla de la suma resta la intersección: sin ese término contaríamos dos veces el resultado 6, que pertenece a A y a B.

---
## 3. Independencia y la falacia del jugador

Dos eventos son **independientes** si saber que uno ha ocurrido no cambia la probabilidad del otro. Con dos dados, "el primero es 6" y "el segundo es 6" lo son, y por tanto P(ambos son 6) = 1/6 · 1/6 = 1/36.

In [ ]:
N = 500_000
d1 = rng.integers(1, 7, size=N)
d2 = rng.integers(1, 7, size=N)
ambos_6 = (d1 == 6) & (d2 == 6)

print(f"Exacto 1/36 = {1/36:.5f}   Simulación = {ambos_6.mean():.5f}")
print(f"P(2º es 6 | 1º es 6) ≈ {(d2[d1 == 6] == 6).mean():.4f}     (1/6 = {1/6:.4f})")

### Experimento guiado: ¿"toca" cruz después de una racha de cruces?

Muchas personas creen que, tras cinco cruces seguidas, la cara está "debida". Lo comprobamos: lanzamos una moneda 1 000 000 de veces, buscamos todas las posiciones que van precedidas por cinco cruces consecutivas y calculamos la frecuencia de cara en el lanzamiento siguiente.

In [ ]:
N = 1_000_000
caras = rng.random(N) < 0.5                       # True = cara
cruces = (~caras).astype(int)

c = np.concatenate(([0], np.cumsum(cruces)))      # c[k] = cruces en los k primeros lanzamientos
racha_5 = (c[5:N] - c[0:N - 5]) == 5              # los 5 lanzamientos anteriores fueron cruz
siguiente = caras[5:]

print(f"Frecuencia global de cara              : {caras.mean():.4f}")
print(f"Posiciones precedidas por 5 cruces       : {racha_5.sum():,}")
print(f"Frecuencia de cara tras una racha de 5 cruces: {siguiente[racha_5].mean():.4f}")

**¿Qué observas?**

- La frecuencia de cara tras cinco cruces, ¿es mayor, menor o igual que la global?
- ¿Qué implica esto sobre la moneda "recordando" lo que ha pasado? Aplícalo a un jugador de ruleta que apuesta al rojo "porque ya han salido cinco negros".

*(Tu respuesta aquí.)*

---
## 4. Cuando NO hay independencia: extraer sin reemplazo

Si sacas dos cartas de una baraja de 52 **sin devolver** la primera, el segundo resultado depende del primero. Probabilidad exacta de que ambas sean ases: 4/52 · 3/51. Multiplicar 4/52 · 4/52 (como si fueran independientes) sería un error.

In [ ]:
N = 100_000
orden = rng.random((N, 52)).argsort(axis=1)[:, :2]      # dos cartas distintas por baraja simulada
ambos_ases = (orden < 4).all(axis=1)                    # las cartas 0-3 son los ases

exacto = (4 / 52) * (3 / 51)
erroneo = (4 / 52) ** 2
print(f"Exacto (sin reemplazo) : {exacto:.5f}")
print(f"Error: tratarlos como independientes : {erroneo:.5f}")
print(f"Simulación ({N:,} barajas)      : {ambos_ases.mean():.5f}")

**Observa:** la simulación coincide con el cálculo exacto y descarta el "atajo" erróneo, que sobrestima la probabilidad en torno a un 30 %.

---
## 5. Probabilidad condicional con una tabla de contingencia

Una tienda registra 1 000 clientes según el canal de contacto y si compraron (*ejemplo ilustrativo*, datos sintéticos). La probabilidad condicional se calcula **restringiendo la tabla** a la fila o columna que nos interesa: P(A | B) = P(A ∩ B) / P(B).

In [ ]:
tabla = pd.DataFrame({"compra": [120, 90], "no compra": [280, 510]},
                     index=["web", "tienda"])
display(tabla)

total = tabla.values.sum()
n_web = tabla.iloc[0].sum()
n_compra = tabla.iloc[:, 0].sum()
n_web_compra = tabla.iloc[0, 0]

print(f"P(compra)                    = {n_compra / total:.3f}")
print(f"P(compra | web)              = {n_web_compra / n_web:.3f}")
print(f"P(compra | tienda)           = {tabla.iloc[1, 0] / tabla.iloc[1].sum():.3f}")
print(f"P(web | compra)              = {n_web_compra / n_compra:.3f}      ← ¡no es P(compra | web)!")

**Observa:** P(compra | web) = 0,30 y P(web | compra) ≈ 0,57 son cosas distintas: la primera responde "de los que entran por web, ¿cuántos compran?", la segunda "de los que compran, ¿cuántos vinieron por web?". Además, como P(compra | web) ≠ P(compra), el canal y la compra **no son independientes**.

---
## 6. Caso guía: un test diagnóstico (frecuencias naturales)

Un test para una enfermedad tiene **sensibilidad** 95 % (detecta al 95 % de los enfermos), **especificidad** 95 % (da negativo al 95 % de los sanos) y la **prevalencia** es del 1 %. *(Todos estos números son un ejemplo ilustrativo.)*

Antes de usar Bayes, hacemos lo más intuitivo: imaginar **10 000 personas** y contar cuántas caen en cada casilla.

In [ ]:
PREVALENCIA = 0.01         # configurable: ejemplo ilustrativo
SENSIBILIDAD = 0.95
ESPECIFICIDAD = 0.95
N_PERSONAS = 10_000

def frecuencias_naturales(prevalencia, sensibilidad, especificidad, n=10_000):
    """Cuántas personas de cada tipo hay entre n."""
    enfermos = n * prevalencia
    sanos = n - enfermos
    vp = enfermos * sensibilidad            # verdaderos positivos
    fn = enfermos - vp                      # falsos negativos
    fp = sanos * (1 - especificidad)        # falsos positivos
    vn = sanos - fp                         # verdaderos negativos
    return {"enfermos": enfermos, "sanos": sanos, "vp": vp, "fn": fn, "fp": fp, "vn": vn}

def vpp_exacto(prevalencia, sensibilidad, especificidad):
    """Valor predictivo positivo: P(enfermo | positivo), por Bayes."""
    numerador = prevalencia * sensibilidad
    return numerador / (numerador + (1 - prevalencia) * (1 - especificidad))

def vpn_exacto(prevalencia, sensibilidad, especificidad):
    """Valor predictivo negativo: P(sano | negativo)."""
    numerador = (1 - prevalencia) * especificidad
    return numerador / (numerador + prevalencia * (1 - sensibilidad))

f = frecuencias_naturales(PREVALENCIA, SENSIBILIDAD, ESPECIFICIDAD, N_PERSONAS)
tabla_nat = pd.DataFrame(
    {"Enfermos": [f["vp"], f["fn"], f["enfermos"]],
     "Sanos": [f["fp"], f["vn"], f["sanos"]]},
    index=["Test positivo", "Test negativo", "Total"])
tabla_nat["Total"] = tabla_nat.sum(axis=1)
display(tabla_nat.round(0).astype(int))

print(f"Positivos totales: {f['vp'] + f['fp']:.0f}  →  de ellos, enfermos de verdad: {f['vp']:.0f}")
print(f"VPP = P(enfermo | positivo) = {vpp_exacto(PREVALENCIA, SENSIBILIDAD, ESPECIFICIDAD):.4f}")
print(f"VPN = P(sano | negativo)     = {vpn_exacto(PREVALENCIA, SENSIBILIDAD, ESPECIFICIDAD):.4f}")

plt.bar(["Test positivo"], [f["vp"]], color="seagreen", label="Verdaderos positivos")
plt.bar(["Test positivo"], [f["fp"]], bottom=[f["vp"]], color="crimson", label="Falsos positivos")
plt.title("¿Quiénes dan positivo? (10 000 personas, ejemplo ilustrativo)")
plt.ylabel("Personas"); plt.legend()
plt.show()

**¿Qué observas?**

- De las 10 000 personas, ¿cuántas dan positivo? ¿Cuántas de ellas están realmente enfermas?
- Un test "preciso al 95 %" acierta el 95 % de las veces en enfermos y en sanos, y sin embargo la mayoría de los positivos son falsos. ¿Qué número de la tabla lo explica?
- Si tu resultado es positivo, ¿qué probabilidad tienes de estar enfermo? Compárala con tu intuición inicial.

*(Tu respuesta aquí.)*

---
## 7. Bayes exacto frente a simulación de Monte Carlo

Simulamos un millón de personas: cada una está enferma con probabilidad igual a la prevalencia y da positivo con probabilidad igual a la sensibilidad (si está enferma) o a 1 − especificidad (si está sana). Estimamos P(enfermo | positivo) como la **fracción de enfermos entre los positivos** y la comparamos con el valor exacto. El error típico de una proporción es √(p(1−p)/n), con n el número de positivos.

In [ ]:
def simular_test(prevalencia, sensibilidad, especificidad, n, rng):
    """Simula n personas: devuelve (enfermo, positivo) como arrays booleanos."""
    enfermo = rng.random(n) < prevalencia
    u = rng.random(n)
    positivo = np.where(enfermo, u < sensibilidad, u < 1 - especificidad)
    return enfermo, positivo

N_MC = 1_000_000
enfermo_mc, positivo_mc = simular_test(PREVALENCIA, SENSIBILIDAD, ESPECIFICIDAD, N_MC, rng)

vpp_ex = vpp_exacto(PREVALENCIA, SENSIBILIDAD, ESPECIFICIDAD)
n_pos = positivo_mc.sum()
vpp_sim = enfermo_mc[positivo_mc].mean()
ee = np.sqrt(vpp_ex * (1 - vpp_ex) / n_pos)

print(f"Positivos en la simulación : {n_pos:,} de {N_MC:,}")
print(f"VPP exacto (Bayes)          : {vpp_ex:.4f}")
print(f"VPP simulado                 : {vpp_sim:.4f}   (error típico ≈ {ee:.4f})")
print(f"Diferencia en errores típicos     : {abs(vpp_sim - vpp_ex) / ee:.2f}")
print()
print(f"P(positivo | enfermo) ≈ {positivo_mc[enfermo_mc].mean():.4f}   ← sensibilidad")
print(f"P(enfermo | positivo) ≈ {vpp_sim:.4f}   ← ¡son cosas muy distintas!")

**¿Qué observas?** ¿A cuántos errores típicos queda la simulación del valor exacto? ¿Qué te dice el último par de líneas sobre confundir P(A|B) con P(B|A)?

Ahora repetimos la simulación con distinto número de personas para ver cómo mejora la estimación.

In [ ]:
filas = []
for n in [1_000, 10_000, 100_000, 1_000_000]:
    enf, pos = simular_test(PREVALENCIA, SENSIBILIDAD, ESPECIFICIDAD, n, rng)
    est = enf[pos].mean()
    filas.append({"personas simuladas": n, "positivos": int(pos.sum()),
                  "VPP simulado": est, "error absoluto": abs(est - vpp_ex)})
display(pd.DataFrame(filas).set_index("personas simuladas").round(4))

**Observa:** con pocas personas simuladas hay muy pocos positivos y la estimación oscila; al multiplicar N por 10 el error baja aproximadamente a la raíz de 10 (≈ 3 veces). Es el mismo fenómeno de la clase 1: la simulación también tiene ruido de muestreo.

---
## 8. El efecto de la prevalencia

¿Qué pasa con el valor predictivo positivo (VPP) si el mismo test (sensibilidad y especificidad fijas) se aplica en poblaciones con distinta prevalencia? Calculamos la curva con la fórmula exacta.

In [ ]:
prevalencias = np.linspace(0.001, 0.5, 500)
for esp in [0.90, 0.95, 0.99]:
    plt.plot(prevalencias, vpp_exacto(prevalencias, 0.95, esp), label=f"especificidad {esp:.0%}")
plt.axvline(PREVALENCIA, color="gray", linestyle=":", label="prevalencia del caso guía (1 %)")
plt.xscale("log")
plt.title("VPP según la prevalencia (sensibilidad 95 %, ejemplo ilustrativo)")
plt.xlabel("Prevalencia (escala log)"); plt.ylabel("VPP = P(enfermo | positivo)"); plt.legend()
plt.show()

filas = [{"prevalencia": p, "VPP": vpp_exacto(p, 0.95, 0.95),
          "falsos por cada 1 000 positivos": 1000 * (1 - vpp_exacto(p, 0.95, 0.95))}
         for p in [0.001, 0.01, 0.05, 0.10, 0.50]]
display(pd.DataFrame(filas).set_index("prevalencia").round(3))

prev_50 = (1 - 0.95) / (0.95 + 1 - 0.95)
print(f"Con sensibilidad = especificidad = 95 %, el VPP llega al 50 % cuando la prevalencia es {prev_50:.0%}.")

**¿Qué observas?**

- ¿Qué le pasa al VPP cuando la enfermedad es muy rara? ¿Y cuando la prevalencia es alta?
- ¿Cuánto ayuda mejorar la especificidad del 95 % al 99 % cuando la prevalencia es baja?
- ¿Por qué se usan los tests de cribado sobre todo en poblaciones de riesgo y no sobre toda la población?

*(Tu respuesta aquí.)*

---
## 9. Un segundo test: la probabilidad se actualiza

Tras un primer positivo, la probabilidad de estar enfermo pasa de la prevalencia (1 %) al VPP (≈ 16 %). Si repetimos un test **independiente** (dado el estado de salud), ese 16 % se convierte en la nueva "prevalencia" de partida. Lo calculamos con Bayes y lo comprobamos sobre las personas que dieron positivo en la simulación anterior.

In [ ]:
vpp_dos_tests = vpp_exacto(vpp_ex, SENSIBILIDAD, ESPECIFICIDAD)      # la probabilidad a posteriori pasa a ser a priori

idx = np.flatnonzero(positivo_mc)                                    # los que dieron positivo en el primer test
u2 = rng.random(idx.size)
positivo_2 = np.where(enfermo_mc[idx], u2 < SENSIBILIDAD, u2 < 1 - ESPECIFICIDAD)
vpp_2_sim = enfermo_mc[idx][positivo_2].mean()

print(f"Tras un positivo              : {vpp_ex:.4f}")
print(f"Tras dos positivos (exacto)     : {vpp_dos_tests:.4f}")
print(f"Tras dos positivos (simulación) : {vpp_2_sim:.4f}")

**Observa:** un segundo positivo eleva la probabilidad de estar enfermo a cerca del 78 %. Pero atención: esto solo vale si los errores de los dos tests son independientes. Si un mismo factor (por ejemplo, una sustancia que interfiere) provoca el falso positivo en ambos, los resultados no son independientes y el cálculo sobrestima el efecto.

---
## Resumen de la clase

1. **Probabilidad:** frecuencia relativa a largo plazo; la simulación converge al valor exacto, con ruido que baja como 1/√n.
2. **Eventos:** el complementario suma 1 y la regla de la suma resta la intersección.
3. **Independencia:** P(A ∩ B) = P(A)·P(B) solo si son independientes; los lanzamientos de una moneda no "se compensan".
4. **Condicional:** P(A|B) = P(A ∩ B)/P(B), y **P(A|B) ≠ P(B|A)**.
5. **Bayes y frecuencias naturales:** con prevalencia baja, incluso un test con 95 % de sensibilidad y especificidad da mayoría de falsos positivos (VPP ≈ 16 % con prevalencia del 1 %, ejemplo ilustrativo).
6. **Monte Carlo:** permite estimar probabilidades simulando; hay que acompañar la estimación de su error típico.

**Siguiente paso:** Clase 5 — Distribuciones de probabilidad y teorema del límite central. Y ahora, a practicar: `notebooks_practica/clase04_practica_probabilidad_analista.ipynb`.